# 05 — O Eco da COVID: O Que Mudou Permanentemente no Sistema?

Os notebooks anteriores demonstraram:
- NB03: 64% do aumento de mortalidade é efeito sistema, não demografia
- NB04: O gap de UTI é majoritariamente confundimento por idade

Se não é simplesmente falta de leitos de UTI, **o que mudou no sistema?** Este notebook investiga:
1. **Mudanças estruturais** — capacidade de leitos, equipamentos, workforce antes vs depois da COVID
2. **Mudanças no fluxo de pacientes** — para onde os pacientes J96 estão indo?
3. **Mudanças no processo de cuidado** — permanência, procedimentos, custos
4. **O efeito rebote** — mortalidade de outras condições também piorou?

**Pergunta de Pesquisa:** RQ4 — A COVID deixou efeitos estruturais permanentes?

**Depende de:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `related_conditions.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_related_conditions,
    load_cnes_names, hospital_name,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
related = load_related_conditions()
names_df = load_cnes_names()

resp = resp.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")
resp["has_icu_days"] = (resp["icu_days"] > 0).astype(int)
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])

pre = resp[resp["covid_era"] == "pre_covid"]
covid = resp[resp["covid_era"] == "covid_acute"]
post_early = resp[resp["covid_era"] == "post_covid_early"]
post_late = resp[resp["covid_era"] == "post_covid_late"]

print(f"Total: {len(resp):,}")
for era in ERA_ORDER:
    n = len(resp[resp['covid_era'] == era])
    m = resp[resp['covid_era'] == era]['MORTE'].mean()
    print(f"  {ERA_LABELS[era]:<35}: {n:>7,} ({m:.1%})")

Total: 116,374
  Pre-COVID (2016–2019)              :  44,447 (31.1%)
  COVID Acute (2020–2021)            :  26,322 (32.4%)
  Post-COVID Early (2022–H1 2023)    :  17,248 (33.2%)
  Post-COVID Late (H2 2023–2025)     :  28,357 (36.3%)


## 1. Fluxo de Pacientes: Para Onde Vão os Pacientes J96?

In [2]:
# Hospital counts treating J96 by era
era_hosp_metrics = []
for era in ERA_ORDER:
    era_data = resp[resp["covid_era"] == era]
    if len(era_data) == 0:
        continue
    era_hosp_metrics.append({
        "era": era,
        "n_admissions": len(era_data),
        "n_hospitals": era_data["CNES"].nunique(),
        "mortality": era_data["MORTE"].mean(),
        "mean_age": era_data["age"].mean(),
        "mean_los": era_data["DIAS_PERM"].mean(),
        "mean_cost": era_data["VAL_TOT"].mean(),
        "pct_no_icu_hosp": (era_data["icu_capacity_level"].astype(str) == "no_icu").mean(),
        "pct_icu_days": era_data["has_icu_days"].mean(),
        "pct_emergency": era_data["is_emergency"].mean(),
        "pct_migrated": era_data["migrated"].mean(),
    })

era_df = pd.DataFrame(era_hosp_metrics)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
metrics_to_plot = [
    ("mortality", "Taxa de Mortalidade", True),
    ("n_admissions", "Volume de Internações", False),
    ("n_hospitals", "Hospitais Ativos", False),
    ("mean_age", "Idade Média", False),
    ("mean_los", "Permanência Média (dias)", False),
    ("mean_cost", "Custo Médio (R$)", False),
    ("pct_icu_days", "% com Dias de UTI", True),
    ("pct_migrated", "% Migrou de Município", True),
]

for ax, (col, title, is_pct) in zip(axes.flat, metrics_to_plot):
    era_colors_list = [ERA_COLORS[e] for e in era_df["era"]]
    vals = era_df[col] * 100 if is_pct else era_df[col]
    bars = ax.bar(range(len(era_df)), vals, color=era_colors_list, alpha=0.85)
    ax.set_xticks(range(len(era_df)))
    ax.set_xticklabels([ERA_LABELS[e].split("(")[0].strip()[:12] for e in era_df["era"]], fontsize=7, rotation=15)
    ax.set_title(title, fontweight="bold", fontsize=10)
    for bar, v in zip(bars, vals):
        fmt = f"{v:.1f}%" if is_pct else (f"{v:,.0f}" if v > 100 else f"{v:.1f}")
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), fmt,
                ha="center", va="bottom", fontsize=7, fontweight="bold")

fig.suptitle("Métricas do Sistema por Era COVID", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "era_system_metrics", prefix="05")

print("\nMétricas do sistema por era:")
print(era_df.to_string(index=False))

  Saved: 05_era_system_metrics.png

Métricas do sistema por era:
             era  n_admissions  n_hospitals  mortality  mean_age  mean_los   mean_cost  pct_no_icu_hosp  pct_icu_days  pct_emergency  pct_migrated
       pre_covid         44447          470   0.311427 41.430603 10.332148 3169.258319         0.482912      0.318424       0.954305      0.237811
     covid_acute         26322          475   0.324026 50.856926  8.154927 2815.606661         0.572221      0.288086       0.949738      0.195768
post_covid_early         17248          450   0.331865 41.895582  9.826357 3636.220667         0.474954      0.349722       0.961619      0.227737
 post_covid_late         28357          459   0.362838 44.486864  9.943330 3730.210541         0.521988      0.363085       0.954403      0.244560


## 2. Hospitais que Pararam vs Começaram a Tratar J96

In [3]:
hosp_pre = set(pre["CNES"].unique())
hosp_covid = set(covid["CNES"].unique())
hosp_post = set(post_late["CNES"].unique())

# Hospitals that stopped treating J96 after COVID
stopped = hosp_pre - hosp_post
started = hosp_post - hosp_pre
continued = hosp_pre & hosp_post

# Get stats for stopped/started
stopped_stats = pre[pre["CNES"].isin(stopped)]
started_stats = post_late[post_late["CNES"].isin(started)]

print(f"Hospitais que tratavam J96 pré-COVID: {len(hosp_pre)}")
print(f"Hospitais que tratam J96 pós-COVID tardio: {len(hosp_post)}")
print(f"  Continuaram: {len(continued)}")
print(f"  Pararam: {len(stopped)} ({len(stopped_stats):,} internações pré-COVID)")
print(f"  Começaram: {len(started)} ({len(started_stats):,} internações pós-COVID tardio)")

if len(stopped_stats) > 0:
    print(f"\nPerfil dos hospitais que PARARAM:")
    print(f"  Mortalidade média: {stopped_stats['MORTE'].mean():.1%}")
    print(f"  Idade média pacientes: {stopped_stats['age'].mean():.0f}")
    print(f"  Volume médio/hospital: {len(stopped_stats)/len(stopped):.0f}")

if len(started_stats) > 0:
    print(f"\nPerfil dos hospitais que COMEÇARAM:")
    print(f"  Mortalidade média: {started_stats['MORTE'].mean():.1%}")
    print(f"  Idade média pacientes: {started_stats['age'].mean():.0f}")
    print(f"  Volume médio/hospital: {len(started_stats)/len(started):.0f}")

# Volume shift for continuing hospitals
cont_pre = pre[pre["CNES"].isin(continued)].groupby("CNES").agg(
    n_pre=("MORTE", "count"), mort_pre=("MORTE", "mean")
).reset_index()
cont_post = post_late[post_late["CNES"].isin(continued)].groupby("CNES").agg(
    n_post=("MORTE", "count"), mort_post=("MORTE", "mean")
).reset_index()
cont = cont_pre.merge(cont_post, on="CNES")
cont["volume_change"] = cont["n_post"] / cont["n_pre"] - 1
cont["mort_change"] = cont["mort_post"] - cont["mort_pre"]

print(f"\nHospitais que continuaram ({len(cont)}):")
print(f"  Volume médio mudou: {cont['volume_change'].median():+.0%}")
print(f"  Mortalidade mediana mudou: {cont['mort_change'].median()*100:+.1f}pp")

Hospitais que tratavam J96 pré-COVID: 470
Hospitais que tratam J96 pós-COVID tardio: 459
  Continuaram: 414
  Pararam: 56 (682 internações pré-COVID)
  Começaram: 45 (1,165 internações pós-COVID tardio)

Perfil dos hospitais que PARARAM:
  Mortalidade média: 30.9%
  Idade média pacientes: 48
  Volume médio/hospital: 12

Perfil dos hospitais que COMEÇARAM:
  Mortalidade média: 39.6%
  Idade média pacientes: 49
  Volume médio/hospital: 26

Hospitais que continuaram (414):
  Volume médio mudou: -32%
  Mortalidade mediana mudou: +0.9pp


## 3. Mudanças no Processo de Cuidado

In [4]:
# LOS distribution comparison by era
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: LOS distribution pre vs post
ax = axes[0]
for era, color in [("pre_covid", ERA_COLORS["pre_covid"]), ("post_covid_late", ERA_COLORS["post_covid_late"])]:
    era_data = resp[resp["covid_era"] == era]
    ax.hist(era_data["DIAS_PERM"].clip(0, 60), bins=60, alpha=0.5, color=color, density=True,
            label=ERA_LABELS[era].split("(")[0].strip())
ax.set_xlabel("Dias de Permanência")
ax.set_ylabel("Densidade")
ax.set_title("A. Distribuição de Permanência", fontweight="bold")
ax.legend(fontsize=8)

# Panel B: mortality by LOS bins
ax = axes[1]
resp["los_bin"] = pd.cut(resp["DIAS_PERM"], bins=[0, 1, 3, 7, 14, 30, 999],
                          labels=["0-1d", "2-3d", "4-7d", "8-14d", "15-30d", ">30d"])
for era, color in [("pre_covid", ERA_COLORS["pre_covid"]), ("post_covid_late", ERA_COLORS["post_covid_late"])]:
    era_data = resp[resp["covid_era"] == era]
    los_mort = era_data.groupby("los_bin", observed=True)["MORTE"].mean()
    ax.plot(range(len(los_mort)), los_mort.values * 100, marker="o", linewidth=2, color=color,
            label=ERA_LABELS[era].split("(")[0].strip())
ax.set_xticks(range(6))
ax.set_xticklabels(["0-1d", "2-3d", "4-7d", "8-14d", "15-30d", ">30d"])
ax.set_ylabel("Mortalidade (%)")
ax.set_title("B. Mortalidade por Faixa de Permanência", fontweight="bold")
ax.legend(fontsize=8)

# Panel C: cost distribution
ax = axes[2]
cost_by_era = resp.groupby("covid_era").agg(
    mean_cost=("VAL_TOT", "mean"),
    median_cost=("VAL_TOT", "median"),
    total_cost=("VAL_TOT", "sum"),
).reindex(ERA_ORDER)
era_colors_list = [ERA_COLORS[e] for e in ERA_ORDER]
bars = ax.bar(range(len(cost_by_era)), cost_by_era["mean_cost"], color=era_colors_list, alpha=0.85)
ax.set_xticks(range(len(cost_by_era)))
ax.set_xticklabels([ERA_LABELS[e].split("(")[0].strip()[:12] for e in ERA_ORDER], fontsize=7, rotation=15)
ax.set_ylabel("Custo Médio (R$)")
ax.set_title("C. Custo Médio por Era", fontweight="bold")

fig.suptitle("Mudanças no Processo de Cuidado", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "care_process_changes", prefix="05")

# LOS and cost by era
print("\nPermanência e custo por era:")
for era in ERA_ORDER:
    d = resp[resp["covid_era"] == era]
    print(f"  {ERA_LABELS[era]:<35}: LOS={d['DIAS_PERM'].mean():.1f}d (med={d['DIAS_PERM'].median():.0f}d), "
          f"Custo=R${d['VAL_TOT'].mean():,.0f} (med=R${d['VAL_TOT'].median():,.0f})")

  Saved: 05_care_process_changes.png

Permanência e custo por era:
  Pre-COVID (2016–2019)              : LOS=10.3d (med=5d), Custo=R$3,169 (med=R$810)
  COVID Acute (2020–2021)            : LOS=8.2d (med=4d), Custo=R$2,816 (med=R$721)
  Post-COVID Early (2022–H1 2023)    : LOS=9.8d (med=5d), Custo=R$3,636 (med=R$869)
  Post-COVID Late (H2 2023–2025)     : LOS=9.9d (med=6d), Custo=R$3,730 (med=R$926)


## 4. Efeito Rebote: Outras Condições Respiratórias

In [5]:
# Compare J96 mortality trend to J18 (pneumonia), J80 (ARDS), J44 (COPD)
if len(related) > 0:
    related["year"] = pd.to_numeric(related["year"], errors="coerce")
    related["MORTE"] = pd.to_numeric(related["MORTE"], errors="coerce").fillna(0).astype(int)
    related = related[related["year"] >= 2016].copy()
    related["condition"] = related["DIAG_PRINC"].astype(str).str[:3]

    # J96 yearly from resp
    j96_yearly = resp.groupby("year").agg(n=("MORTE", "count"), mortality=("MORTE", "mean")).reset_index()
    j96_yearly["condition"] = "J96"

    # Other conditions yearly
    other_yearly = related.groupby(["year", "condition"]).agg(
        n=("MORTE", "count"), mortality=("MORTE", "mean")
    ).reset_index()

    all_conditions = pd.concat([j96_yearly, other_yearly], ignore_index=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    cond_colors = {"J96": COLORS["danger"], "J18": COLORS["primary"], "J80": COLORS["warning"], "J44": COLORS["success"]}
    cond_labels = {"J96": "J96 Insuf. Resp.", "J18": "J18 Pneumonia", "J80": "J80 SDRA", "J44": "J44 DPOC"}

    for cond in ["J96", "J18", "J80", "J44"]:
        grp = all_conditions[all_conditions["condition"] == cond].sort_values("year")
        if len(grp) > 0:
            ax1.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2.5,
                    label=cond_labels.get(cond, cond), color=cond_colors.get(cond, COLORS["muted"]))
            # Normalize volume to 2016 = 100
            base = grp[grp["year"]==2016]["n"].values
            if len(base) > 0 and base[0] > 0:
                ax2.plot(grp["year"], grp["n"] / base[0] * 100, marker="o", linewidth=2,
                        label=cond_labels.get(cond, cond), color=cond_colors.get(cond, COLORS["muted"]))

    ax1.axvspan(2020, 2021.5, alpha=0.06, color="red")
    ax1.set_title("A. Mortalidade por Condição Respiratória", fontweight="bold")
    ax1.set_ylabel("Mortalidade (%)")
    ax1.legend(fontsize=9)

    ax2.axvspan(2020, 2021.5, alpha=0.06, color="red")
    ax2.axhline(100, color="black", linestyle=":", alpha=0.3)
    ax2.set_title("B. Volume Indexado (2016 = 100)", fontweight="bold")
    ax2.set_ylabel("Índice")
    ax2.legend(fontsize=9)

    fig.suptitle("J96 vs Outras Condições Respiratórias — O Aumento É Específico?", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_plot(fig, "cross_condition_comparison", prefix="05")

    # Mortality change pre vs post for each condition
    print("\nVariação de mortalidade por condição (pré-COVID vs pós-COVID tardio):")
    for cond in ["J96", "J18", "J80", "J44"]:
        grp = all_conditions[all_conditions["condition"] == cond]
        pre_yrs = grp[grp["year"].isin([2016, 2017, 2018, 2019])]
        post_yrs = grp[grp["year"].isin([2024, 2025])]
        if len(pre_yrs) > 0 and len(post_yrs) > 0:
            m_pre = np.average(pre_yrs["mortality"], weights=pre_yrs["n"])
            m_post = np.average(post_yrs["mortality"], weights=post_yrs["n"])
            print(f"  {cond_labels.get(cond, cond):<25}: {m_pre:.1%} → {m_post:.1%} ({(m_post-m_pre)*100:+.1f}pp)")
else:
    print("Related conditions data not available")

  Saved: 05_cross_condition_comparison.png

Variação de mortalidade por condição (pré-COVID vs pós-COVID tardio):
  J96 Insuf. Resp.         : 31.0% → 36.4% (+5.4pp)
  J18 Pneumonia            : 12.3% → 13.9% (+1.5pp)
  J80 SDRA                 : 20.1% → 19.9% (-0.2pp)
  J44 DPOC                 : 11.9% → 12.2% (+0.3pp)


## 5. Mudanças na Concentração Hospitalar

In [6]:
# Gini coefficient of hospital concentration by era
def gini(values):
    values = np.sort(np.array(values, dtype=float))
    n = len(values)
    if n == 0 or values.sum() == 0:
        return 0
    index = np.arange(1, n + 1)
    return (2 * (index * values).sum() / (n * values.sum())) - (n + 1) / n

concentration = []
for era in ERA_ORDER:
    era_data = resp[resp["covid_era"] == era]
    hosp_vol = era_data["CNES"].value_counts().values
    g = gini(hosp_vol)
    # Top 10 hospital share
    top10_share = era_data["CNES"].value_counts().head(10).sum() / len(era_data)
    concentration.append({"era": era, "gini": g, "top10_share": top10_share,
                          "n_hospitals": era_data["CNES"].nunique()})

conc_df = pd.DataFrame(concentration)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

era_colors_list = [ERA_COLORS[e] for e in conc_df["era"]]
ax1.bar(range(len(conc_df)), conc_df["gini"], color=era_colors_list, alpha=0.85)
ax1.set_xticks(range(len(conc_df)))
ax1.set_xticklabels([ERA_LABELS[e].split("(")[0].strip()[:12] for e in conc_df["era"]], fontsize=8, rotation=15)
ax1.set_title("A. Coeficiente de Gini (Concentração Hospitalar)", fontweight="bold")
ax1.set_ylabel("Gini")

ax2.bar(range(len(conc_df)), conc_df["top10_share"] * 100, color=era_colors_list, alpha=0.85)
ax2.set_xticks(range(len(conc_df)))
ax2.set_xticklabels([ERA_LABELS[e].split("(")[0].strip()[:12] for e in conc_df["era"]], fontsize=8, rotation=15)
ax2.set_title("B. % de Internações nos Top 10 Hospitais", fontweight="bold")
ax2.set_ylabel("%")

fig.suptitle("Concentração Hospitalar por Era", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "concentration_by_era", prefix="05")

print("\nConcentração por era:")
for _, r in conc_df.iterrows():
    print(f"  {ERA_LABELS[r['era']]:<35}: Gini={r['gini']:.3f}, Top10={r['top10_share']:.1%}, Hospitais={r['n_hospitals']}")

  Saved: 05_concentration_by_era.png

Concentração por era:
  Pre-COVID (2016–2019)              : Gini=0.701, Top10=26.2%, Hospitais=470
  COVID Acute (2020–2021)            : Gini=0.683, Top10=18.8%, Hospitais=475
  Post-COVID Early (2022–H1 2023)    : Gini=0.688, Top10=29.4%, Hospitais=450
  Post-COVID Late (H2 2023–2025)     : Gini=0.675, Top10=23.5%, Hospitais=459


## 6. Análise de Mortalidade por Natureza Jurídica e Era

In [7]:
# Mortality by legal nature and era
nat_era = resp.groupby(["covid_era", "nat_jur_category"]).agg(
    n=("MORTE", "count"), mortality=("MORTE", "mean")
).reset_index()

main_nats = ["public", "filantropica", "assoc_privada", "private"]
nat_labels = {"public": "Público", "filantropica": "Filantrópica", "assoc_privada": "Assoc. Privada", "private": "Privado"}

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(ERA_ORDER))
width = 0.18
nat_colors = [COLORS["primary"], COLORS["success"], COLORS["warning"], COLORS["secondary"]]

for i, (nat, color) in enumerate(zip(main_nats, nat_colors)):
    vals = []
    for era in ERA_ORDER:
        row = nat_era[(nat_era["covid_era"]==era) & (nat_era["nat_jur_category"]==nat)]
        vals.append(row["mortality"].values[0] * 100 if len(row) > 0 else np.nan)
    ax.bar(x + i*width, vals, width, label=nat_labels.get(nat, nat), color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([ERA_LABELS[e].split("(")[0].strip() for e in ERA_ORDER], fontsize=9)
ax.set_ylabel("Mortalidade (%)")
ax.set_title("Mortalidade por Natureza Jurídica × Era COVID", fontsize=14, fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
save_plot(fig, "legal_nature_by_era", prefix="05")

print("\nMortalidade por natureza jurídica e era:")
for nat in main_nats:
    print(f"  {nat_labels.get(nat, nat):<18}:", end="")
    for era in ERA_ORDER:
        row = nat_era[(nat_era["covid_era"]==era) & (nat_era["nat_jur_category"]==nat)]
        if len(row) > 0:
            print(f"  {row['mortality'].values[0]:.1%}", end="")
        else:
            print("    -  ", end="")
    print()

  Saved: 05_legal_nature_by_era.png

Mortalidade por natureza jurídica e era:
  Público           :  33.0%  36.0%  34.6%  39.4%
  Filantrópica      :    -      -      -      -  
  Assoc. Privada    :  31.2%  35.3%  34.2%  35.4%
  Privado           :  8.2%  24.2%  19.0%  9.4%


## 7. Síntese e Métricas

In [8]:
metrics = {
    "era_metrics": era_df.to_dict(orient="records"),
    "hospitals_pre": len(hosp_pre),
    "hospitals_post": len(hosp_post),
    "hospitals_stopped": len(stopped),
    "hospitals_started": len(started),
    "hospitals_continued": len(continued),
    "concentration_gini": conc_df.to_dict(orient="records"),
    "mortality_pre_covid": round(pre["MORTE"].mean(), 4),
    "mortality_post_late": round(post_late["MORTE"].mean(), 4),
    "mortality_change_pp": round((post_late["MORTE"].mean() - pre["MORTE"].mean()) * 100, 1),
    "los_pre_covid": round(pre["DIAS_PERM"].mean(), 1),
    "los_post_late": round(post_late["DIAS_PERM"].mean(), 1),
}

save_metrics(metrics, "05_covid_echo")

print("\n" + "=" * 70)
print("SÍNTESE — O QUE MUDOU PERMANENTEMENTE")
print("=" * 70)
print(f"\n1. MORTALIDADE: {pre['MORTE'].mean():.1%} → {post_late['MORTE'].mean():.1%} ({(post_late['MORTE'].mean()-pre['MORTE'].mean())*100:+.1f}pp)")
print(f"2. VOLUME: {len(pre)/4:,.0f}/ano → {len(post_late)/2.5:,.0f}/ano")
print(f"3. HOSPITAIS: {len(hosp_pre)} → {len(hosp_post)} ({len(stopped)} pararam, {len(started)} começaram)")
print(f"4. PERMANÊNCIA: {pre['DIAS_PERM'].mean():.1f}d → {post_late['DIAS_PERM'].mean():.1f}d")
print(f"5. % COM UTI DIAS: {pre['has_icu_days'].mean():.1%} → {post_late['has_icu_days'].mean():.1%}")
print(f"6. CONCENTRAÇÃO: Gini {conc_df[conc_df['era']=='pre_covid']['gini'].values[0]:.3f} → {conc_df[conc_df['era']=='post_covid_late']['gini'].values[0]:.3f}")

  Saved metrics: 05_covid_echo.json

SÍNTESE — O QUE MUDOU PERMANENTEMENTE

1. MORTALIDADE: 31.1% → 36.3% (+5.1pp)
2. VOLUME: 11,112/ano → 11,343/ano
3. HOSPITAIS: 470 → 459 (56 pararam, 45 começaram)
4. PERMANÊNCIA: 10.3d → 9.9d
5. % COM UTI DIAS: 31.8% → 36.3%
6. CONCENTRAÇÃO: Gini 0.701 → 0.675
